# Galileo Brain V2 — Fine-Tuning Qwen3-4B con Unsloth

**10 celle**. Copia-incolla una per una. Dopo la cella 1, riavvia il runtime.

Miglioramenti V2 vs V1:
1. Dataset 4x più grande: 2000 esempi (era 500)
2. LoRA rank 64 (era 32) — più capacità espressiva
3. alpha=128 (era 32) — ratio 2:1 per stronger learning
4. batch_size=4 con gradient_accumulation=4 — effective batch 16
5. Warmup 30 steps (era 10) — più stabile con dataset più grande
6. Train/eval split 90/10 per monitorare overfitting
7. save_strategy + eval ogni 25 steps
8. Cella evaluation suite integrata

In [ ]:
# ============================================================
# CELLA 1 — Install (ESATTA da notebook ufficiale Unsloth)
# ============================================================
# DOPO QUESTA CELLA: Runtime > Restart session
# ============================================================

%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {
        '2.10': '0.0.34',
        '2.9': '0.0.33.post1',
        '2.8': '0.0.32.post2'
    }.get(v, '0.0.34')
    !pip install sentencepiece protobuf \"datasets==4.3.0\" \"huggingface_hub>=0.34.0\" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

print(\"--- Install completato. Riavvia il runtime (Runtime > Restart session) ---\")

In [ ]:
# ============================================================
# CELLA 2 — Carica modello + LoRA V2 (rank=64, alpha=128)
# ============================================================
# CHECKPOINT: deve stampare Trainable > 0
# ============================================================

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=\"unsloth/Qwen3-4B-Instruct-2507\",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,                    # V2: era 32
    target_modules=[\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\",
                     \"gate_proj\", \"up_proj\", \"down_proj\"],
    lora_alpha=128,          # V2: era 32 (ratio 2:1)
    lora_dropout=0.05,       # V2: era 0 (leggero dropout per generalizzazione)
    bias=\"none\",
    use_gradient_checkpointing=\"unsloth\",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f\"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)\")
assert trainable > 0, \"ERRORE: 0 parametri trainabili!\"
print(f\"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB\")

In [ ]:
# ============================================================
# CELLA 3 — Upload dataset V2
# ============================================================

from google.colab import files
import os

DATASET_PATH = \"galileo-brain-v2.jsonl\"
if not os.path.exists(DATASET_PATH):
    print(\"Carica galileo-brain-v2.jsonl (7.1 MB, 2000 esempi)...\")
    uploaded = files.upload()
    DATASET_PATH = list(uploaded.keys())[0]

with open(DATASET_PATH) as f:
    num_lines = sum(1 for _ in f)
print(f\"Dataset: {DATASET_PATH} \u2014 {num_lines} esempi\")

In [ ]:
# ============================================================
# CELLA 4 — Prepara dataset con train/eval split (90/10)
# ============================================================

from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template=\"qwen3-instruct\")

full_dataset = load_dataset(\"json\", data_files={\"train\": DATASET_PATH}, split=\"train\")
print(f\"Colonne: {full_dataset.column_names}, Righe: {len(full_dataset)}\")

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples[\"messages\"]:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {\"text\": texts}

full_dataset = full_dataset.map(formatting_prompts_func, batched=True)

# V2: Train/eval split 90/10
split = full_dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = split[\"train\"]
eval_dataset = split[\"test\"]

print(f\"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}\")
print(f\"\\nEsempio formattato (500 chars):\\n{train_dataset[0]['text'][:500]}\")

In [ ]:
# ============================================================
# CELLA 5 — SFTTrainer V2 + train_on_responses_only
# ============================================================
# V2: batch_size=4, grad_accum=4, warmup=30, eval ogni 25 steps
# ============================================================

from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field=\"text\",
        per_device_train_batch_size=4,     # V2: era 2
        gradient_accumulation_steps=4,
        warmup_steps=30,                   # V2: era 10
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        eval_strategy=\"steps\",            # V2: monitoraggio overfitting
        eval_steps=25,
        optim=\"adamw_8bit\",
        weight_decay=0.01,
        lr_scheduler_type=\"cosine\",
        seed=3407,
        report_to=\"none\",
        output_dir=\"galileo-brain-v2-output\",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,       # V2: carica il migliore
        metric_for_best_model=\"eval_loss\",
    ),
)

# Masking: loss calcolata SOLO sulle risposte assistant
trainer = train_on_responses_only(
    trainer,
    instruction_part=\"<|im_start|>user\\n\",
    response_part=\"<|im_start|>assistant\\n\",
)

print(\"Trainer V2 pronto. Loss solo su risposte assistant.\")
print(f\"Steps stimati: {len(train_dataset) // (4 * 4) * 3} (3 epochs)\")

In [ ]:
# ============================================================
# CELLA 6 — Training V2
# ============================================================
# CHECKPOINT: loss finale deve scendere sotto ~0.1
# eval_loss deve restare vicina a train_loss (no overfitting)
# ============================================================

import time

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f\"\\n{'='*50}\")
print(f\"TRAINING V2 COMPLETATO!\")
print(f\"Tempo: {elapsed/60:.1f} min\")
print(f\"Loss finale: {stats.training_loss:.4f}\")
print(f\"GPU peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB\")

In [ ]:
# ============================================================
# CELLA 7 — Test inferenza (6 test, tutti i 6 intent)
# ============================================================
# CHECKPOINT: i 6 test devono produrre JSON valido
# ============================================================

import json
FastLanguageModel.for_inference(model)

# System prompt dal dataset
with open(DATASET_PATH) as f:
    first = json.loads(f.readline())
SYSTEM = first[\"messages\"][0][\"content\"]

def test(msg):
    text = tokenizer.apply_chat_template(
        [{\"role\": \"system\", \"content\": SYSTEM},
         {\"role\": \"user\", \"content\": msg}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors=\"pt\").to(\"cuda\")
    out = model.generate(
        **inputs, max_new_tokens=512,
        temperature=0.1, top_p=0.95,
    )
    resp = tokenizer.decode(
        out[0][inputs[\"input_ids\"].shape[-1]:],
        skip_special_tokens=True
    )
    try:
        return json.loads(resp)
    except:
        return {\"raw\": resp}

# V2: test tutti e 6 gli intent
tests = [
    (\"action\", \"[CONTESTO]\\ntab: simulator\\ncomponenti: [led1]\\nfili: 1\\n\\n[MESSAGGIO]\\nAvvia la simulazione\"),
    (\"circuit\", \"[CONTESTO]\\ntab: simulator\\ncomponenti: []\\nfili: 0\\n\\n[MESSAGGIO]\\nMetti un LED e un resistore\"),
    (\"tutor\",   \"[CONTESTO]\\ntab: simulator\\ncomponenti: [resistor1]\\n\\n[MESSAGGIO]\\nCos'e un resistore?\"),
    (\"code\",    \"[CONTESTO]\\ntab: editor\\ncomponenti: [led1]\\n\\n[MESSAGGIO]\\nScrivi il codice per il blink\"),
    (\"vision\",  \"[CONTESTO]\\ntab: simulator\\ncomponenti: [led1, resistor1]\\nfili: 2\\n\\n[MESSAGGIO]\\nGuarda il mio circuito\"),
    (\"nav\",     \"[CONTESTO]\\ntab: simulator\\n\\n[MESSAGGIO]\\nApri il manuale\"),
]

passed = 0
for expected, t in tests:
    r = test(t)
    intent = r.get(\"intent\", \"?\") if isinstance(r, dict) else \"?\"
    ok = \"raw\" not in r
    if ok: passed += 1
    symbol = \"\u2705\" if ok else \"\u26a0\ufe0f\"
    match = \"\u2705\" if intent.startswith(expected[:3]) else \"\u274c\"
    print(f\"{symbol} expected={expected:8s} got={intent:12s} {match} | {str(r)[:100]}\")

print(f\"\\n{passed}/6 test passati\")

In [ ]:
# ============================================================
# CELLA 8 — Evaluation Suite (20 test rapidi)
# ============================================================
# Subset della evaluation suite completa (120 test)
# ============================================================

import json

eval_tests = [
    {\"input\": \"Avvia la simulazione\", \"exp_intent\": \"action\"},
    {\"input\": \"Stop\", \"exp_intent\": \"action\"},
    {\"input\": \"Resetta tutto\", \"exp_intent\": \"action\"},
    {\"input\": \"Cancella tutto dalla breadboard\", \"exp_intent\": \"action\"},
    {\"input\": \"Metti un LED sulla breadboard\", \"exp_intent\": \"circuit\"},
    {\"input\": \"Aggiungi un resistore\", \"exp_intent\": \"circuit\"},
    {\"input\": \"Costruisci un circuito con LED e resistore\", \"exp_intent\": \"circuit\"},
    {\"input\": \"Collega il LED al pin D3\", \"exp_intent\": \"circuit\"},
    {\"input\": \"Carica l'esperimento del primo LED\", \"exp_intent\": \"navigation\"},
    {\"input\": \"Apri il manuale\", \"exp_intent\": \"navigation\"},
    {\"input\": \"Vai al volume 2\", \"exp_intent\": \"navigation\"},
    {\"input\": \"Compila il codice\", \"exp_intent\": \"action\"},
    {\"input\": \"Scrivi il codice per il blink del LED\", \"exp_intent\": \"code\"},
    {\"input\": \"Come funziona un LED?\", \"exp_intent\": \"tutor\"},
    {\"input\": \"Spiegami la legge di Ohm\", \"exp_intent\": \"tutor\"},
    {\"input\": \"Ciao Galileo!\", \"exp_intent\": \"tutor\"},
    {\"input\": \"Guarda il mio circuito\", \"exp_intent\": \"vision\"},
    {\"input\": \"Analizza la mia breadboard\", \"exp_intent\": \"vision\"},
    {\"input\": \"Rimuovi il LED\", \"exp_intent\": \"action\"},
    {\"input\": \"Premi il pulsante\", \"exp_intent\": \"action\"},
]

ctx = \"[CONTESTO]\\ntab: simulator\\nesperimento: v1-cap6-esp1\\ncomponenti: [led1, resistor1, pushbutton1]\\nfili: 3\\nvolume_attivo: 1\"

correct = 0
for i, t in enumerate(eval_tests, 1):
    msg = f\"{ctx}\\n\\n[MESSAGGIO]\\n{t['input']}\"
    r = test(msg)
    intent = r.get(\"intent\", \"?\") if isinstance(r, dict) else \"?\"
    ok = intent == t[\"exp_intent\"]
    if ok: correct += 1
    sym = \"\u2705\" if ok else \"\u274c\"
    print(f\"{sym} [{i:2d}] exp={t['exp_intent']:10s} got={intent:10s} | {t['input'][:50]}\")

pct = correct / len(eval_tests) * 100
print(f\"\\n{'='*50}\")
print(f\"EVAL SCORE: {correct}/{len(eval_tests)} ({pct:.0f}%)\")
if pct >= 90: print(\"\u2705 PASS\")
elif pct >= 75: print(\"\u26a0\ufe0f PARTIAL\")
else: print(\"\u274c FAIL\")

In [ ]:
# ============================================================
# CELLA 9 — Salva LoRA + GGUF
# ============================================================

# LoRA adapter
model.save_pretrained(\"galileo-brain-v2-lora\")
tokenizer.save_pretrained(\"galileo-brain-v2-lora\")
print(\"LoRA V2 salvato.\")

# GGUF per Ollama
model.save_pretrained_gguf(
    \"galileo-brain-v2-gguf\", tokenizer,
    quantization_method=\"q4_k_m\",
)
print(\"GGUF q4_k_m V2 salvato.\")

In [ ]:
# ============================================================
# CELLA 10 — Download GGUF
# ============================================================

from google.colab import files
import os

gguf_dir = \"galileo-brain-v2-gguf\"
for f in os.listdir(gguf_dir):
    if f.endswith(\".gguf\"):
        size = os.path.getsize(os.path.join(gguf_dir, f)) / 1e9
        print(f\"Scaricando {f} ({size:.2f} GB)...\")
        files.download(os.path.join(gguf_dir, f))